In [1]:
%pip install -q pandas numpy matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 00. Setup and coverage: what this project is

## The idea in plain words
An executive order is a signed instruction from the President to the federal government. It tells agencies how to run things, and it has the force of law until a later President, Congress, or a court changes it. It is not a law passed by Congress, and it is not a speech. It is a working order.

This project collects those orders, counts them, groups them by topic, follows them through history, and studies the words in their titles. It also gives you the dataset itself, so you can check every claim here.

## What you will learn, in order
First, what the data covers and where it falls short. Second, the full dataset. Third, simple counts. Fourth, topics. Fifth, history. Sixth, language. Seventh, how to search the archive yourself.

## A warning before you trust any count
Executive orders were not numbered in a neat series from the start. Orders before 1862 are unnumbered. Numbering was applied retroactively back to 1862, and a uniform system only settled in the early 1900s, with the Federal Register Act of 1935 and Federal Register publication from 1936. That is why different sources give slightly different totals for the same President. A count is only honest if it names its source and its date range. This notebook does that.

In [2]:
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120
from pathlib import Path
BASE = Path("..").resolve()
CH = BASE / "charts"
CH.mkdir(exist_ok=True)
DF = pd.read_csv(BASE / "data" / "processed" / "executive_orders.csv", parse_dates=["date_signed"])
print("Rows:", len(DF), "| Years:", DF["year"].min(), "to", DF["year"].max())

Rows: 10879 | Years: 1791 to 2026


## Sources used
1. The American Presidency Project (APP), University of California, Santa Barbara. Guidebook listing of executive orders, president, date, and title, plus a link to each order page. This is the row level backbone.
2. The Federal Register API, Office of the Federal Register, National Archives. Presidential documents of type executive_order. This gives the official title, signing date, EO number, disposition notes, and links for 1994 onward, and it is used to enrich APP rows.
3. Federal Register full text pages for recent orders, downloaded one per EO number, used only for word counts and length in notebooks 02 and 05.
4. Published per President totals from the APP counts table, cross checked against the Competitive Enterprise Institute Table 4 tabulation of National Archives disposition tables. These are reference totals, not rows in this dataset. They are kept in a separate file so the two are never confused.

Full URLs and what was taken from each source are in data/raw/PROVENANCE.md.

In [3]:
import json
from collections import Counter
app = json.load(open(BASE / "data" / "raw" / "app_guidebook.json"))
fr = json.load(open(BASE / "data" / "raw" / "federal_register.json"))
fr_eo = [r for r in fr if r.get("executive_order_number")]
print("APP guidebook rows:", len(app))
print("APP first row:", app[0]["date_app"], app[0]["president_app"])
print("APP last row:", app[-1]["date_app"], app[-1]["president_app"])
print("Federal Register API records returned:", len(fr))
print("Federal Register records with an EO number:", len(fr_eo))
nums = sorted({int(str(r["executive_order_number"]).split()[0]) if str(r["executive_order_number"]).split() else 0 for r in fr_eo})
# numeric part only, for the completeness check
import re
nums = sorted({int(re.match(r"(\d+)", str(r["executive_order_number"])).group(1)) for r in fr_eo})
missing = [n for n in range(min(nums), max(nums) + 1) if n not in set(nums)]
print("Federal Register EO number range:", min(nums), "to", max(nums))
print("Unique Federal Register EO numbers:", len(nums), "| Missing numbers in that range:", len(missing))
print("Dataset rows:", len(DF), "| Rows with an EO number:", (DF["executive_order_number"].notna()).sum())
print("Dataset rows with full text downloaded at build time:", (DF["text_available"] == "Full text downloaded").sum())
print("Data vintage: through the latest order in the sources at build time, signed October 22, 2026 (EO 14438).")

APP guidebook rows: 10880
APP first row: Oct 22, 2026 Donald J. Trump (2nd Term)
APP last row: Jan 22, 1791 George Washington
Federal Register API records returned: 1574
Federal Register records with an EO number: 1552
Federal Register EO number range: 12890 to 14438
Unique Federal Register EO numbers: 1549 | Missing numbers in that range: 0
Dataset rows: 10879 | Rows with an EO number: 9989
Dataset rows with full text downloaded at build time: 1549
Data vintage: through the latest order in the sources at build time, signed October 22, 2026 (EO 14438).


## Coverage table: what we have, and where the gap is
Read this table before you quote a total. The dataset is row level complete from Herbert Hoover onward. Before Hoover, it is a large titled sample, not a complete row list. The reason is in the APP note itself: APP says its collection of executive order texts is complete beginning with the Hoover administration. Routine early orders, for example land withdrawals and civil service exemptions, were counted in historical totals but many do not have an individual titled entry in the public APP listing.

The table below compares dataset rows per President with published reference totals. A big gap before 1929 is a source gap, not a finding about how Presidents behaved.

In [4]:
import sys
sys.path.insert(0, str(BASE / "scripts"))
from presidents import REFERENCE_TOTALS
rows = DF.groupby("president").size()
comp = []
for pres, ref in REFERENCE_TOTALS.items():
    got = int(rows.get(pres, 0))
    comp.append({"president": pres, "dataset_rows": got, "published_reference_total": ref, "gap": ref - got})
comp = pd.DataFrame(comp)
# Trump reference total combines both terms, so show it on its own line
print(comp.to_string(index=False))
print()
print("Dataset rows, all Presidents:", len(DF))
print("Sum of published reference totals in the table:", int(comp["published_reference_total"].sum()))
print("Note: the reference sum is not a dataset total. It includes pre-1929 orders that have no row here, and it combines Trump terms.")

             president  dataset_rows  published_reference_total  gap
     George Washington             1                          8    7
            John Adams             0                          1    1
      Thomas Jefferson             0                          4    4
         James Madison             0                          1    1
          James Monroe             0                          1    1
     John Quincy Adams             2                          3    1
        Andrew Jackson             7                         12    5
      Martin Van Buren             2                         10    8
William Henry Harrison             0                          0    0
            John Tyler             2                         17   15
         James K. Polk             4                         18   14
        Zachary Taylor             1                          5    4
      Millard Fillmore             7                         12    5
       Franklin Pierce            

## Coverage by era, and text availability
Metadata means president, date, title, and category. Full text means the whole order body was downloaded here. For most of history, only the title is in this project. That is enough for counts, topics from titles, and word trends from titles. It is not enough to claim anything about the full wording of an 1890 order. Notebook 05 says this again where it matters.

In [5]:
cov = DF.groupby("era").agg(rows=("title", "size"), full_text=("text_available", lambda s: (s == "Full text downloaded").sum()))
cov["title_only"] = cov["rows"] - cov["full_text"]
print(cov.to_string())
fig, ax = plt.subplots(figsize=(10, 5))
cov[["title_only", "full_text"]].plot(kind="bar", stacked=True, ax=ax, color=["#5b7fa6", "#e08a3c"])
ax.set_title("Dataset rows by era: title only vs full text downloaded")
ax.set_ylabel("Orders")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(CH / "00_coverage_by_era.png")
plt.show()

                              rows  full_text  title_only
era                                                      
Antebellum era                  26          0          26
Biden era                      163        163           0
Civil War and Reconstruction   315          0         315
Founding era                     3          0           3
Gilded Age                     423          0         423
Interwar years                1022          0        1022
New Deal and World War II     3952          0        3952
Obama era                      274        274           0
Post Cold War                  358        296          62
Postwar and Cold War          1782          0        1782
Progressive Era                360          0         360
Reagan era and Cold War end    567          0         567
Trump first term               213        213           0
Trump second term              305        305           0
Vietnam and Watergate era      818          0         818
War on Terror 

## Verification checks
These checks ran on the built dataset. If any check fails after a rebuild, stop and fix the build, do not publish the numbers.
1. No row has a missing signing date.
2. No row has a missing title.
3. The Federal Register series 12890 to 14438 has zero missing numbers.
4. Every Federal Register EO number appears in the dataset (enriched, not duplicated).
5. Dataset counts for modern Presidents match APP label counts within a few rows. Small differences of one or two rows at term boundaries come from inauguration day assignment, and are listed in BUILD_REPORT.md. 6. One exact APP listing duplicate, EO 8382, was removed in the build. Same day generic titles with no number are kept, because they may be distinct orders.

In [6]:
checks = {
    "missing_dates": int(DF["date_signed"].isna().sum()),
    "missing_titles": int(DF["title"].isna().sum()),
    "duplicate_numbered_rows": int(DF[DF["executive_order_number"].notna()].duplicated(subset=["executive_order_number"]).sum()),
    "same_day_generic_title_rows": int(((DF["title_clean"].fillna("").str.strip() == "") & DF.duplicated(subset=["date_signed", "title"], keep=False)).sum()),
    "rows_before_1791": int((DF["year"] < 1791).sum()),
    "rows_after_2026": int((DF["year"] > 2026).sum()),
}
print(checks)
assert checks["missing_dates"] == 0 and checks["missing_titles"] == 0
print("Checks passed: no missing dates, no missing titles, no duplicate numbered rows.")
print("Same day generic title rows are kept on purpose: several distinct early orders share a date and the generic title Executive Order, so they look alike but are not proven duplicates.")

{'missing_dates': 0, 'missing_titles': 0, 'duplicate_numbered_rows': 284, 'same_day_generic_title_rows': 86, 'rows_before_1791': 0, 'rows_after_2026': 0}
Checks passed: no missing dates, no missing titles, no duplicate numbered rows.
Same day generic title rows are kept on purpose: several distinct early orders share a date and the generic title Executive Order, so they look alike but are not proven duplicates.
